In [1]:
import numpy as np

# Simple environment (5 states, 2 actions)
n_states = 5
n_actions = 2

Q = np.zeros((n_states, n_actions))

alpha = 0.1
gamma = 0.9
epsilon = 0.1

def step(state, action):
    next_state = (state + action + 1) % n_states
    reward = 1 if next_state == 4 else 0
    return next_state, reward

# Training
for episode in range(500):
    state = np.random.randint(n_states)

    for _ in range(20):
        if np.random.rand() < epsilon:
            action = np.random.randint(n_actions)
        else:
            action = np.argmax(Q[state])

        next_state, reward = step(state, action)

        Q[state, action] += alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )

        state = next_state

print("Q-Table:\n", Q)

Q-Table:
 [[2.98892989 2.73800408]
 [3.32103321 3.32103312]
 [3.32102804 3.6900369 ]
 [3.6900369  2.15741634]
 [2.6900368  2.98892989]]


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.net(x)

model = DQN()
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

gamma = 0.9

for episode in range(300):
    state = np.array([np.random.randint(5)], dtype=np.float32)

    for _ in range(20):
        state_tensor = torch.tensor(state)
        q_values = model(state_tensor)

        action = torch.argmax(q_values).item()

        next_state = np.array([(state[0] + action + 1) % 5], dtype=np.float32)
        reward = 1 if next_state[0] == 4 else 0

        target = q_values.clone().detach()
        target[action] = reward + gamma * torch.max(model(torch.tensor(next_state)))

        loss = loss_fn(q_values, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state

print("DQN trained ✔️")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\Harsh\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        return self.net(x)

policy = Policy()
optimizer = optim.Adam(policy.parameters(), lr=0.01)

gamma = 0.9

for episode in range(300):
    log_probs = []
    rewards = []

    state = np.array([np.random.randint(5)], dtype=np.float32)

    for _ in range(20):
        probs = policy(torch.tensor(state))
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()

        log_probs.append(dist.log_prob(action))

        next_state = np.array([(state[0] + action.item() + 1) % 5], dtype=np.float32)
        reward = 1 if next_state[0] == 4 else 0

        rewards.append(reward)
        state = next_state

    # Compute returns
    G = 0
    returns = []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)

    loss = 0
    for log_prob, G in zip(log_probs, returns):
        loss -= log_prob * G

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Policy Gradient trained ✔️")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\Harsh\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.